**Set environment**

In [1]:
source ../run_config_project.sh
show_env

BASE DIRECTORY (FD_BASE):      /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO):      /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK):      /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA):      /hpc/group/igvf/kk319/data
CONTAINER DIR. (FD_SING):      /hpc/group/igvf/kk319/container

You are working with           
PATH OF PROJECT (FD_PRJ):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references
PR

## Preview

In [2]:
ls -1 ${FD_RES}

analysis_variant_delta_alphagenome
analysis_variant_motif_richard
predict_variant_alphagenome
predict_variant_kircher2019


In [3]:
ls ${FD_RES}/analysis_variant_motif_richard

background_zero_order.npy
background_zero_order.tsv
batches_dev
batches_full
batches_pilot
JASPAR2024_CORE_vertebrates_non-redundant.lods.npz
JASPAR2024_CORE_vertebrates_non-redundant.lods.pkl
JASPAR2024_CORE_vertebrates_non-redundant.pmap.npz
JASPAR2024_CORE_vertebrates_non-redundant.pmap.pkl
motifdelta_pilot_jaspar2024
motifdelta_pilot_jvierstra_v2.0beta
motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
motif_jaspar2024_core_vertebrates_nonredundant.pmap.pkl
motif_nonredundant_jvierstra_v2.0beta.lods.pkl
motif_nonredundant_jvierstra_v2.0beta.pmap.pkl
tmp
variant_closed_gof_bluestarr_flank35_obs.fa
variant_closed_gof_bluestarr_flank35_ref.fa
variant_closed_gof_bluestarr_flank35_unobs.fa
variant_closed_gof_bluestarr_flankL35R70_obs.fa
variant_closed_gof_bluestarr_flankL35R70_ref.fa
variant_closed_gof_bluestarr_flankL35R70_unobs.fa
variant_closed_gof_bluestarr.tsv.gz
variant_closed_gof_bluestarr_withseq_flank35.tsv.gz
variant_closed_gof_bluestarr_withseq_flankL35R70.tsv.gz


In [4]:
ls ${FD_RES}/analysis_variant_motif_richard/batches_pilot

variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_obs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_ref.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_unobs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002_obs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002_ref.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002_unobs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003_obs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003_ref.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003_unobs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk004_obs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk004_ref.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk004_unobs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk005_obs.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk005_ref.fa
variant_closed_gof_bluestarr_flankL35R70_pilot_chunk005_unobs.fa
variant_closed_gof_bluestarr_flankL35R70_pilo

## Execute

### Jaspar2024

In [12]:
FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_pilot
FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
FP_MODEL=${FD_RES}/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.pmap.pkl
FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_pilot_jaspar2024
#mkdir -p "${FD_DELTA}"

In [13]:
### set resource
NUM_CPU=5
NUM_MEM=20G

### loop each chunk and run motifdelta
for idx in $(seq 1 20); do

    ### prepare
    CHUNK=$(printf "chunk%03d" "${idx}")
    PREFIX=variant_closed_gof_bluestarr_flankL35R70_pilot_${CHUNK}
    echo "Running motif scan for ${PREFIX}..."

    ### execute
    FP_EXE=${FD_EXE}/run_motifdelta_01_scan.py
    FN_LOG=run_motifdelta_scan_batch_pilot_jaspar_${CHUNK}.txt
    FP_LOG=${FD_LOG}/${FN_LOG}
    echo '${FD_LOG}'/"${FN_LOG}"
    
    sbatch \
        -A majoroslab \
        -p igvf,common \
        --cpus-per-task ${NUM_CPU} \
        --mem    ${NUM_MEM} \
        --output ${FP_LOG}  \
        --chdir  ${FD_EXE}  \
        ${FP_APP} python ${FP_EXE} \
            --txt_fpath_fasta_ref  ${FD_BATCH}/${PREFIX}_ref.fa \
            --txt_fpath_fasta_obs  ${FD_BATCH}/${PREFIX}_obs.fa \
            --txt_fpath_fasta_ubs  ${FD_BATCH}/${PREFIX}_unobs.fa \
            --txt_fpath_motif      ${FP_MOTIF} \
            --txt_fpath_output     ${FD_DELTA}/${PREFIX}_scan.npz
done

Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk001.txt
Submitted batch job 40271066
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk002.txt
Submitted batch job 40271067
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk003.txt
Submitted batch job 40271068
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk004...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk004.txt
Submitted batch job 40271069
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk005...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk005.txt
Submitted batch job 40271070
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk006...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chun

### jvierstra non-redundant motif

In [21]:
FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_pilot
FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.0beta.lods.pkl
FP_MODEL=${FD_RES}/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.0beta.pmap.pkl
FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_pilot_jvierstra_v2.0beta
#mkdir -p "${FD_DELTA}"

In [22]:
### set resource
NUM_CPU=5
NUM_MEM=20G

### loop each chunk and run motifdelta
for idx in $(seq 1 20); do

    ### prepare
    CHUNK=$(printf "chunk%03d" "${idx}")
    PREFIX=variant_closed_gof_bluestarr_flankL35R70_pilot_${CHUNK}
    echo "Running motif scan for ${PREFIX}..."

    ### execute
    FP_EXE=${FD_EXE}/run_motifdelta_01_scan.py
    FN_LOG=run_motifdelta_scan_batch_pilot_jvierstra_${CHUNK}.txt
    FP_LOG=${FD_LOG}/${FN_LOG}
    echo '${FD_LOG}'/"${FN_LOG}"
    
    sbatch \
        -A majoroslab \
        -p igvf,common \
        --cpus-per-task ${NUM_CPU} \
        --mem    ${NUM_MEM} \
        --output ${FP_LOG}  \
        --chdir  ${FD_EXE}  \
        ${FP_APP} python ${FP_EXE} \
            --txt_fpath_fasta_ref  ${FD_BATCH}/${PREFIX}_ref.fa \
            --txt_fpath_fasta_obs  ${FD_BATCH}/${PREFIX}_obs.fa \
            --txt_fpath_fasta_ubs  ${FD_BATCH}/${PREFIX}_unobs.fa \
            --txt_fpath_motif      ${FP_MOTIF} \
            --txt_fpath_output     ${FD_DELTA}/${PREFIX}_scan.npz
done

Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jvierstra_chunk001.txt
Submitted batch job 40271430
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jvierstra_chunk002.txt
Submitted batch job 40271431
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jvierstra_chunk003.txt
Submitted batch job 40271432
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk004...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jvierstra_chunk004.txt
Submitted batch job 40271433
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk005...
${FD_LOG}/run_motifdelta_scan_batch_pilot_jvierstra_chunk005.txt
Submitted batch job 40271434
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk006...
${FD_LOG}/run_motifdelta_scan_batch_pi

## Review

### JASPAR 2024

In [14]:
cat ${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk001.txt

Loading FASTA sequences...
Loaded 5000 sequences
Load and check complete in 0.06 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 14.34 seconds
Estimated memory use (ref+obs+unobs): 7.269 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz
Saved complete in 26.28 seconds

Done.


In [17]:
JOBIDS=40271066
sacct_summary.sh ${JOBIDS}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40271066.ba+                          batch  COMPLETED   00:00:42  00:38.067     15112M 

===== ElapsedRaw =====
ElapsedRaw = 42 sec (0.70 min)

===== MaxRSS =====
MaxRSS = 14.76 GiB


In [16]:
cat ${FD_LOG}/run_motifdelta_scan_batch_pilot_jaspar_chunk020.txt

Loading FASTA sequences...
Loaded 5000 sequences
Load and check complete in 0.08 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 91.04 seconds
Estimated memory use (ref+obs+unobs): 7.269 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot_jaspar2024/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk020_scan.npz
Saved complete in 33.15 seconds

Done.


In [18]:
JOBIDS=40271085
sacct_summary.sh ${JOBIDS}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40271085.ba+                          batch  COMPLETED   00:02:08  01:58.141  15469988K 

===== ElapsedRaw =====
ElapsedRaw = 128 sec (2.13 min)

===== MaxRSS =====
MaxRSS = 14.75 GiB


In [20]:
JOBIDS=$(seq 40271066 40271085 | paste -sd' ')
sacct_summary.sh ${JOBIDS}

Detected multiple job IDs (20). Running batch summary.
40271066,40271067,40271068,40271069,40271070,40271071,40271072,40271073,40271074,40271075,40271076,40271077,40271078,40271079,40271080,40271081,40271082,40271083,40271084,40271085
===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40271066.ba+                          batch  COMPLETED   00:00:42  00:38.067     15112M 
40271067.ba+                          batch  COMPLETED   00:00:41  00:37.990  15477028K 
40271068.ba+                          batch  COMPLETED   00:02:03  00:46.506  15473804K 
40271069.ba+                          batch  COMPLETED   00:02:02  00:45.630  15478112K 
40271070.ba+                          batch  COMPLETED   00:01:49  00:43.629  15474344K 
40271071.ba+                          batch  COMPLETED   00:02:03  00:46.290  15476036K 
40271072.ba+         

### jvierstra non-redundant motif

In [24]:
JOBIDS=$(seq 40271430 40271449 | paste -sd' ')
sacct_summary.sh ${JOBIDS}

Detected multiple job IDs (20). Running batch summary.
40271430,40271431,40271432,40271433,40271434,40271435,40271436,40271437,40271438,40271439,40271440,40271441,40271442,40271443,40271444,40271445,40271446,40271447,40271448,40271449
===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40271430.ba+                          batch  COMPLETED   00:01:45  00:42.322  13057384K 
40271431.ba+                          batch  COMPLETED   00:01:44  00:43.025  13057000K 
40271432.ba+                          batch  COMPLETED   00:01:45  00:41.517  13053844K 
40271433.ba+                          batch  COMPLETED   00:01:45  00:40.800  13055200K 
40271434.ba+                          batch  COMPLETED   00:01:42  00:41.353  13058152K 
40271435.ba+                          batch  COMPLETED   00:01:44  00:41.716  13053640K 
40271436.ba+         

```
for JobID in $(seq 40150927 40150948); do
    sacct -j ${JobID} --format=JobID,JobName%30,State,Elapsed,TotalCPU,MaxRSS \
    | grep '\.ba'
done
```

In [16]:
JOBIDS=40166308
sacct_summary.sh ${JOBIDS}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40166308.ba+                          batch  COMPLETED   00:01:58  00:47.316  15475320K 

===== ElapsedRaw =====
ElapsedRaw = 118 sec (1.97 min)

===== MaxRSS =====
MaxRSS = 14.76 GiB


In [18]:
JOBIDS=$(seq 40166308 40166328 | paste -sd' ')
echo ${JOBIDS}

40166308 40166309 40166310 40166311 40166312 40166313 40166314 40166315 40166316 40166317 40166318 40166319 40166320 40166321 40166322 40166323 40166324 40166325 40166326 40166327 40166328


In [24]:
JOBIDS=$(seq 40166308 40166328 | paste -sd' ')
JOBID=$(echo ${JOBIDS[0]} | tr ' ' ',')
echo ${JOBID}

40166308,40166309,40166310,40166311,40166312,40166313,40166314,40166315,40166316,40166317,40166318,40166319,40166320,40166321,40166322,40166323,40166324,40166325,40166326,40166327,40166328


In [23]:
JOBIDS=$(seq 40166308 40166328 | paste -sd' ')
sacct_summary.sh ${JOBIDS}

Detected multiple job IDs (21). Running batch summary.
40166308
===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40166308.ba+                          batch  COMPLETED   00:01:58  00:47.316  15475320K 

===== ElapsedRaw =====
n=1  min=118 sec (2.0 min)  mean=118.0 sec (2.0 min)  max=118 sec (2.0 min)

===== MaxRSS =====
n=1  min=14.76 GiB  mean=14.76 GiB  max=14.76 GiB


In [8]:
JobIDS=$(seq 40166308 40166328 | paste -sd, -)
echo ${JobIDS}

40166308,40166309,40166310,40166311,40166312,40166313,40166314,40166315,40166316,40166317,40166318,40166319,40166320,40166321,40166322,40166323,40166324,40166325,40166326,40166327,40166328


In [26]:
JOBIDS=$(seq 40166308 40166328 | paste -sd' ')
sacct_summary_batch.sh ${JobIDS}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40166308.ba+                          batch  COMPLETED   00:01:58  00:47.316  15475320K 
40166309.ba+                          batch  COMPLETED   00:01:56  00:47.819  15476952K 
40166310.ba+                          batch  COMPLETED   00:01:54  00:43.897  15470564K 
40166311.ba+                          batch  COMPLETED   00:01:57  00:47.623  15472516K 
40166312.ba+                          batch  COMPLETED   00:01:57  00:47.697  15472272K 
40166313.ba+                          batch  COMPLETED   00:01:58  00:49.020  15473072K 
40166314.ba+                          batch  COMPLETED   00:01:56  00:46.472  15472940K 
40166315.ba+                          batch  COMPLETED   00:01:53  00:43.290  15474320K 
40166316.ba+                          batch  COMPLETED   00:00:46  00:41.182  

In [6]:
JobIDS=$(seq 40166308 40166328 | paste -sd, -)
sacct_summary_batch.sh ${JobIDS}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40166308.ba+                          batch  COMPLETED   00:01:58  00:47.316  15475320K 
40166309.ba+                          batch  COMPLETED   00:01:56  00:47.819  15476952K 
40166310.ba+                          batch  COMPLETED   00:01:54  00:43.897  15470564K 
40166311.ba+                          batch  COMPLETED   00:01:57  00:47.623  15472516K 
40166312.ba+                          batch  COMPLETED   00:01:57  00:47.697  15472272K 
40166313.ba+                          batch  COMPLETED   00:01:58  00:49.020  15473072K 
40166314.ba+                          batch  COMPLETED   00:01:56  00:46.472  15472940K 
40166315.ba+                          batch  COMPLETED   00:01:53  00:43.290  15474320K 
40166316.ba+                          batch  COMPLETED   00:00:46  00:41.182  

In [7]:
JobIDS=40166308
sacct_summary_single.sh ${JobIDS}

===== Summary (.ba tasks) =====
JobID                               JobName      State    Elapsed   TotalCPU     MaxRSS 
------------ ------------------------------ ---------- ---------- ---------- ---------- 
40166308.ba+                          batch  COMPLETED   00:01:58  00:47.316  15475320K 

===== ElapsedRaw =====
ElapsedRaw = 118 sec (1.97 min)

===== MaxRSS =====
MaxRSS = 14.76 GiB


In [22]:
JobIDS=$(seq 40166308 40166328 | paste -sd, -)
echo ${JobIDS}
sacct -j ${JobIDS} --format=JobID,JobName%30,State,Elapsed,TotalCPU,MaxRSS \
| grep '\.ba'

sacct -j ${JobIDS} --format=JobID,ElapsedRaw \
| grep '\.ba' \
| awk '
    {
        t = $2                          # ElapsedRaw in seconds
        if (NR == 1 || t < min) min = t # get min
        if (NR == 1 || t > max) max = t # get max 
        sum += t                        # get sum
        n++                             # get number of row
    }
    END {
        mean = (n > 0) ? sum / n : 0 
        printf "n=%d  min=%d sec (%.2f min)  mean=%.2f sec (%.2f min)  max=%d sec (%.2f min)\n",
                n, min, min/60, mean, mean/60, max, max/60
    }
'

sacct -j ${JobIDS} --format=JobID,MaxRSS \
| grep '\.ba' \
| awk '
{
    raw = $2             # e.g. "15480360K"
    unit = substr(raw, length(raw), 1)
    gsub(/[^0-9]/, "", raw)
    val = raw + 0        # numeric value

    # Convert to GiB
    if (unit == "K") {
        gib = val / 1024 / 1024      # K -> GiB
    } else if (unit == "M") {
        gib = val / 1024             # MiB -> GiB
    } else if (unit == "G") {
        gib = val                    # already GiB (ish)
    } else {
        # fallback: assume K if no unit
        gib = val / 1024 / 1024
    }

    if (NR == 1 || gib < min) min = gib
    if (NR == 1 || gib > max) max = gib
    sum += gib
    n++
}
END {
    mean = (n > 0) ? sum / n : 0
    printf "n=%d  min=%.2f GiB  mean=%.2f GiB  max=%.2f GiB\n",
            n, min, mean, max
}'

40166308,40166309,40166310,40166311,40166312,40166313,40166314,40166315,40166316,40166317,40166318,40166319,40166320,40166321,40166322,40166323,40166324,40166325,40166326,40166327,40166328
40166308.ba+                          batch  COMPLETED   00:01:58  00:47.316  15475320K 
40166309.ba+                          batch  COMPLETED   00:01:56  00:47.819  15476952K 
40166310.ba+                          batch  COMPLETED   00:01:54  00:43.897  15470564K 
40166311.ba+                          batch  COMPLETED   00:01:57  00:47.623  15472516K 
40166312.ba+                          batch  COMPLETED   00:01:57  00:47.697  15472272K 
40166313.ba+                          batch  COMPLETED   00:01:58  00:49.020  15473072K 
40166314.ba+                          batch  COMPLETED   00:01:56  00:46.472  15472940K 
40166315.ba+                          batch  COMPLETED   00:01:53  00:43.290  15474320K 
40166316.ba+                          batch  COMPLETED   00:00:46  00:41.182  15504004K 
40166317.b

In [50]:
cat ${FD_LOG}/run_motifdelta_scan_batch_pilot_001.txt

Loading FASTA sequences...
Loaded 5000 sequences
Load and check complete in 0.07 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/JASPAR2024_CORE_vertebrates_non-redundant.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 18.78 seconds
Estimated memory use (ref+obs+unobs): 7.269 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz
Saved complete in 97.83 seconds

Done.


In [20]:
### set resource
NUM_CPU=5
NUM_MEM=20G

### loop each chunk and run motifdelta
for idx in $(seq 1 3); do

    ### prepare
    CHUNK=$(printf "%03d" "${idx}")
    PREFIX=variant_closed_gof_bluestarr_flankL35R70_pilot_chunk${CHUNK}
    echo "Running motif scan for ${PREFIX}..."

    ### execute
    FP_EXE=${FD_EXE}/run_motifdelta_01_scan.py
    FP_LOG=${FD_LOG}/run_motifdelta_scan_batch_pilot_${CHUNK}.txt
    echo '${FD_LOG}'/"run_motifdelta_scan_batch_pilot_${CHUNK}.txt"
    
    sbatch \
        -A majoroslab \
        -p igvf \
        --cpus-per-task ${NUM_CPU} \
        --mem    ${NUM_MEM} \
        --output ${FP_LOG}  \
        --chdir  ${FD_EXE}  \
        ${FP_APP} python ${FP_EXE} \
            --txt_fpath_fasta_ref  ${FD_BATCH}/${PREFIX}_ref.fa \
            --txt_fpath_fasta_obs  ${FD_BATCH}/${PREFIX}_obs.fa \
            --txt_fpath_fasta_ubs  ${FD_BATCH}/${PREFIX}_unobs.fa \
            --txt_fpath_motif      ${FP_MOTIF} \
            --txt_fpath_output     ${FD_DELTA}/${PREFIX}_scan.npz
done

Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001...
${FD_LOG}/run_motifdelta_scan_batch_pilot_001.txt
Submitted batch job 40150536
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002...
${FD_LOG}/run_motifdelta_scan_batch_pilot_002.txt
Submitted batch job 40150538
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003...
${FD_LOG}/run_motifdelta_scan_batch_pilot_003.txt
Submitted batch job 40150539


In [21]:
cat ${FD_LOG}/run_motifdelta_scan_batch_pilot_001.txt

Loading FASTA sequences...
Loaded 5000 sequences

Load and check complete in 0.06 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/JASPAR2024_CORE_vertebrates_non-redundant.lods.pkl

Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 15.15 seconds

Estimated memory use (ref+obs+unobs): 7.269 GB
Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz
Saved complete in 29.19 seconds

Done.


In [22]:
sacct -j 40150536 --format=JobID,MaxRSS,Elapsed

JobID            MaxRSS    Elapsed 
------------ ---------- ---------- 
40150536                  00:00:45 
40150536.ba+  15474556K   00:00:45 
40150536.ex+       256K   00:00:45 


In [23]:
ls -l /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz

-rw-r--r--. 1 kk319 majoroslab 7806022286 Nov 25 11:43 /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz


In [17]:
cat ${FD_LOG}/run_motifdelta_scan_batch_pilot_001.txt

Loading FASTA sequences...
Loaded 5000 sequences

Load and check complete in 0.06 seconds

Loading motif matrices...
Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/JASPAR2024_CORE_vertebrates_non-redundant.lods.pkl

Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 15.88 seconds

Estimated memory use (ref+obs+unobs): 7.269 GB
Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz
Saved complete in 355.87 seconds

Done.


In [18]:
sacct -j 40149712 --format=JobID,MaxRSS,Elapsed

JobID            MaxRSS    Elapsed 
------------ ---------- ---------- 
40149712                  00:06:13 
40149712.ba+  14622544K   00:06:13 
40149712.ex+          0   00:06:13 


In [19]:
ls -l /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz

-rw-r--r--. 1 kk319 majoroslab 6909817975 Nov 25 11:28 /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz


In [24]:
7 806 022 286
6909817975

bash: 7806022286: command not found


: 127

In [10]:
### set resource
NUM_CPU=5
NUM_MEM=20G

### loop each chunk and run motifdelta
for idx in $(seq 1 20); do

    ### prepare
    CHUNK=$(printf "%03d" "${idx}")
    PREFIX=variant_closed_gof_bluestarr_flankL35R70_pilot_chunk${CHUNK}
    echo "Running motif scan for ${PREFIX}..."

    ### execute
    FP_EXE=${FD_EXE}/run_motifdelta_01_scan.py
    FP_LOG=${FD_LOG}/run_motifdelta_scan_batch_pilot_${CHUNK}.txt
    echo '${FD_LOG}'/"run_motifdelta_scan_batch_pilot_${CHUNK}.txt"
    
    sbatch \
        -A majoroslab \
        -p igvf \
        --cpus-per-task ${NUM_CPU} \
        --mem    ${NUM_MEM} \
        --output ${FP_LOG}  \
        --chdir  ${FD_EXE}  \
        ${FP_APP} python ${FP_EXE} \
            --txt_fpath_fasta_ref  ${FD_BATCH}/${PREFIX}_ref.fa \
            --txt_fpath_fasta_obs  ${FD_BATCH}/${PREFIX}_obs.fa \
            --txt_fpath_fasta_ubs  ${FD_BATCH}/${PREFIX}_unobs.fa \
            --txt_fpath_motif      ${FP_MOTIF} \
            --txt_fpath_output     ${FD_DELTA}/${PREFIX}_scan.npz
done

Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001...
${FD_LOG}/run_motifdelta_scan_batch_pilot_001.txt
Submitted batch job 40149247
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk002...
${FD_LOG}/run_motifdelta_scan_batch_pilot_002.txt
Submitted batch job 40149248
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk003...
${FD_LOG}/run_motifdelta_scan_batch_pilot_003.txt
Submitted batch job 40149249
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk004...
${FD_LOG}/run_motifdelta_scan_batch_pilot_004.txt
Submitted batch job 40149253
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk005...
${FD_LOG}/run_motifdelta_scan_batch_pilot_005.txt
Submitted batch job 40149254
Running motif scan for variant_closed_gof_bluestarr_flankL35R70_pilot_chunk006...
${FD_LOG}/run_motifdelta_scan_batch_pilot_006.txt
Submitted batch job 40149255
Running motif scan for variant_clo

## Review

In [13]:
cat ${FD_LOG}/run_motifdelta_scan_batch_pilot_001.txt

Loaded 5000 sequences

Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/JASPAR2024_CORE_vertebrates_non-redundant.lods.pkl

Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Running motif scanning...
Scan complete in 18.14 seconds

Estimated memory use (ref+obs+unobs): 7.269 GB
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz
Done.


In [15]:
sacct -j 40149247 --format=JobID,MaxRSS,Elapsed

JobID            MaxRSS    Elapsed 
------------ ---------- ---------- 
40149247                  00:06:18 
40149247.ba+  14626648K   00:06:18 
40149247.ex+          0   00:06:18 


In [14]:
sacct -j 40149117 --format=JobID,MaxRSS,Elapsed

JobID            MaxRSS    Elapsed 
------------ ---------- ---------- 
40149117                  00:00:35 
40149117.ba+  12581728K   00:00:35 
40149117.ex+       256K   00:00:35 


In [ ]:
Loaded 5000 sequences

Loaded 879 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/JASPAR2024_CORE_vertebrates_non-redundant.lods.pkl

Forward kernels shape: (879, 33, 4)
Reverse kernels shape: (879, 33, 4)
Running motif scanning...
Scan complete in 18.14 seconds

Estimated memory use (ref+obs+unobs): 7.269 GB
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_scan.npz
Done.